<a href="https://colab.research.google.com/github/iespinozahDM/UBA-DM/blob/main/Clase_02_Preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Lectura y escritura de datos

In [16]:
df = pd.read_csv("https://data.insideairbnb.com/argentina/ciudad-aut%C3%B3noma-de-buenos-aires/buenos-aires/2025-01-29/visualisations/listings.csv", index_col="id", parse_dates=["last_review"])
df.head()

,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
id,,,,,,,,,,,,,,,,,
11508,Amazing Luxurious Apt-Palermo Soho,42762,Candela,NaN,Palermo,-34.581840,-58.424150,Entire home/apt,67518.0,3,44,2025-01-26,0.29,1,300,5,NaN
14222,"RELAX IN HAPPY HOUSE - PALERMO, BUENOS AIRES",87710233,María,NaN,Palermo,-34.586170,-58.410360,Entire home/apt,22375.0,7,123,2025-01-18,0.80,6,44,8,NaN
15074,ROOM WITH RIVER SIGHT,59338,Monica,NaN,Nuñez,-34.538920,-58.465990,Private room,NaN,29,0,NaT,NaN,1,0,0,NaN
16695,DUPLEX LOFT 2 - SAN TELMO,64880,Elbio Mariano,NaN,Monserrat,-34.614390,-58.376110,Entire home/apt,52511.0,2,45,2019-11-30,0.27,9,365,0,NaN
20062,PENTHOUSE /Terrace & pool /City views /2bedrooms,75891,Sergio,NaN,Palermo,-34.581848,-58.441605,Entire home/apt,113360.0,2,330,2025-01-17,1.84,4,209,25,NaN


In [4]:
df.dtypes

,0
name,object
host_id,int64
host_name,object
neighbourhood_group,float64
neighbourhood,object
latitude,float64
longitude,float64
room_type,object
price,float64
minimum_nights,int64


In [5]:
df["last_review"].dt.day

,last_review
id,
11508,26.0
14222,18.0
15074,NaN
16695,30.0
20062,17.0
...,...
1343797906145861177,NaN
1343805212193119512,NaN
1343811295921195058,NaN


# Limpieza de columnas

In [6]:
df["name"].str.replace("★", "").str.replace("Condo", "Condominium")

,name
id,
11508,Amazing Luxurious Apt-Palermo Soho
14222,"RELAX IN HAPPY HOUSE - PALERMO, BUENOS AIRES"
15074,ROOM WITH RIVER SIGHT
16695,DUPLEX LOFT 2 - SAN TELMO
20062,PENTHOUSE /Terrace & pool /City views /2bedrooms
...,...
1343797906145861177,Moderno departamento 1 amb dividido balcón
1343805212193119512,Amplias Habitaciones con Terraza en Palermo Soho
1343811295921195058,Loft para 6 personas en Palermo Soho -o82


In [8]:
df["license"] = df["license"].fillna("no license")
#df["license"].fillna("no license", inplace=True)

In [ ]:
# quitar columnas: 2 formas equivalentes
df = df.drop(columns=["content_info_short"])

df = df[df.columns.drop("license")]

# Discretización

In [17]:
df["minimum_nights"].describe()

,minimum_nights
count,35172.000000
mean,6.159871
std,26.072002
min,1.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,1000.000000


In [18]:
# bins por cantidad (q)
pd.qcut(df["minimum_nights"], q=3, labels=['short', 'medium', 'long']).value_counts()

,count
minimum_nights,
short,18168
long,9147
medium,7857


In [19]:
df.minimum_nights.describe()

,minimum_nights
count,35172.000000
mean,6.159871
std,26.072002
min,1.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,1000.000000


In [ ]:
# bins a mano
pd.cut(df["minimum_nights"], bins=[df["minimum_nights"].min(), 2, 4, df["minimum_nights"].max()], labels=['short', 'medium', 'long'])

# Numeración

In [ ]:
# mapeo
df["room_type"].map({"Entire home/apt": 1, "Private room": 2}) # si no está pone NaN // cambia tipo de dato

In [ ]:
# dummies
pd.get_dummies(df, columns=['neighbourhood']) # , dummy_na=True

# Normalización

In [ ]:
(df["minimum_nights"] - df["minimum_nights"].min()) / (df["minimum_nights"].max() - df["minimum_nights"].min())

# Estandarización

In [ ]:
(df["minimum_nights"] - df["minimum_nights"].mean()) / df["minimum_nights"].std()

# Miscelánea

In [ ]:
df["a_estrenar"] = df.name.str.lower().str.contains("a estrenar")

In [ ]:
for p in ["sum", "mascotas"]:
    df[p] = df.name.str.lower().str.contains(p)

In [ ]:
for b in [1,2,3,4,5]:
    df.loc[(df.bathrooms.isna() & df["description"].str.lower().str.contains(f"{b} baño")), "bathroom"] = b

df.loc[(df.bathrooms.isna() & df["description"].str.lower().str.contains(f"un baño")), "bathroom"] = 1
df.loc[(df.bathrooms.isna() & df["description"].str.lower().str.contains(f"dos baño")), "bathroom"] = 2
